<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part1_pixels/01_images_as_data/notebooks/01_images_as_data_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 01: Images as Data · Solutions

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a short comment. There is usually more than one good way to solve an exercise, and your numbers may differ a little from run to run, so read the comments as "what to look for" rather than exact results.

To a computer, a photo is a grid of numbers. In this notebook you open images in Python, look at
the numbers underneath, change them, and see what happens to the picture. By the end you will
have turned a few hundred official portraits into a small dataset and taken a first look at it.

In this notebook you will:

- load an image and see how it is stored: pixels, channels and color spaces
- edit images by changing their numbers, and run into the classic beginner's trap on the way
- see what resolution and JPEG compression do to an image
- read the metadata hidden in image files, and remove it
- build a small collection of portraits, view it as a contact sheet, and compute the "average
  face" of the US Congress
- think about which parts of your own research questions a computer can measure in pixels

**Data.** We use the official portraits of members of the US Congress, which are in the public
domain, from the [unitedstates/images](https://github.com/unitedstates/images) project, and the
matching biographical data from
[unitedstates/congress-legislators](https://github.com/unitedstates/congress-legislators). They
are a good first dataset: free to use, similar in format, and linked to information about the
people in them.

This notebook runs on a normal CPU runtime.

## 0. Setup

Everything we need is preinstalled in Colab. **Pillow** (imported as `PIL`) opens and saves
images, **NumPy** holds their pixels as arrays, and **Matplotlib** shows them.

We also define two small helpers. Whenever we download something, we send a short `User-Agent`
header that says who is asking, which is polite and sometimes required, and we stop with a clear
error if the download fails.

In [ ]:
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from PIL import Image, ImageOps, ExifTags

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}

def download(url):
    """Download a URL and return the response, or raise an error if it fails."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response

def load_image(url):
    """Download an image and open it with Pillow."""
    return Image.open(io.BytesIO(download(url).content))

# Official portraits, by size ("original", "450x550" or "225x275") and Bioguide ID
PHOTO_URL = "https://raw.githubusercontent.com/unitedstates/images/gh-pages/congress/{size}/{bioguide}.jpg"

print("Setup complete!")

## 1. An image is a grid of numbers

Let's open one portrait: Senator Maria Cantwell of Washington, who has served in the Senate since
2001. Every member of Congress has a Bioguide ID, a stable identifier that links the portrait to
other datasets. Hers is `C000127`.

In [ ]:
portrait = load_image(PHOTO_URL.format(size="450x550", bioguide="C000127"))

print("Size (width, height):", portrait.size)
print("Mode:", portrait.mode)

plt.figure(figsize=(3.5, 4.3))
plt.imshow(portrait)
plt.axis("off")
plt.show()

Pillow tells us the image is 450 pixels wide and 550 pixels high, in **RGB** mode: every pixel
has a red, a green and a blue value. To see the numbers themselves, we convert the image to a
NumPy array.

In [ ]:
arr = np.asarray(portrait)

print("Shape:", arr.shape)
print("Data type:", arr.dtype)
print("Smallest and largest value:", arr.min(), arr.max())
print("Pixel at row 100, column 50:", arr[100, 50])

Three things to notice:

1. The **shape** is `(550, 450, 3)`: rows first, then columns, then the three color **channels**.
   Pillow reports size as (width, height) and NumPy as (height, width). Mixing these up is one of
   the most common bugs in image code.
2. The data type is `uint8`, an unsigned 8-bit integer. Every value is a whole number from 0 (no
   light) to 255 (full intensity). Three channels of 256 levels give about 16.7 million colors.
3. A single pixel is just three numbers. `[R, G, B] = [0, 0, 0]` is black, `[255, 255, 255]` is
   white, and `[255, 0, 0]` is pure red.

Zooming in makes the grid visible. Here is a 12 × 12 patch at the corner of the mouth, shown
without smoothing, followed by the red values of its top-left corner.

In [ ]:
patch = arr[330:342, 262:274]

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(patch, interpolation="nearest")
ax.set_title("12 × 12 pixels, enlarged")
ax.axis("off")
plt.show()

print("Red values of the top-left 5 × 5 pixels:")
print(patch[:5, :5, 0])

> **✏️ Exercise 1**
>
> 1. Print the RGB values of the pixel in the exact center of `arr`. *(Hint: `arr.shape` gives you
>    the height and width; `//` divides and rounds down.)*
> 2. Compute the **average** red, green and blue value over the whole image. Which channel is
>    strongest, and does that match what you see in the portrait?
>    *(Hint: `arr.reshape(-1, 3)` turns the image into one row per pixel.)*

In [ ]:
# ✅ Solution
height, width, _ = arr.shape
print("Center pixel:", arr[height // 2, width // 2])

channel_means = arr.reshape(-1, 3).mean(axis=0)
for name, value in zip(["red", "green", "blue"], channel_means):
    print(f"Average {name:<5}: {value:6.1f}")

# The answer depends on the portrait: here the backdrop and the clothing take up most of the
# frame, so they dominate the average far more than the face does. That is worth remembering
# for later: a summary of "the image" is often mostly a summary of its background.

## 2. Channels and color spaces

Each channel on its own is a grayscale image: bright where that color is strong, dark where it is
weak.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 4))
axes[0].imshow(arr)
axes[0].set_title("RGB")
for i, name in enumerate(["red", "green", "blue"]):
    axes[i + 1].imshow(arr[:, :, i], cmap="gray", vmin=0, vmax=255)
    axes[i + 1].set_title(f"{name} channel")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

RGB is how screens produce color, but it is not always the most useful way to *measure* color.
Two alternatives come up again and again in research:

- **Grayscale** (Pillow mode `"L"`, for luminance) keeps only brightness. It is a weighted average
  of the channels, with green weighted most because our eyes are most sensitive to it:
  `L = 0.299 R + 0.587 G + 0.114 B`.
- **HSV** splits a color into **hue** (which color: red, yellow, blue...), **saturation** (how
  vivid, from gray to pure color) and **value** (how bright). If you want to know whether an image
  is colorful or bright regardless of *which* colors it uses, HSV makes that a single channel.
  In Pillow, all three HSV channels run from 0 to 255.

In [ ]:
gray = portrait.convert("L")
hsv = np.asarray(portrait.convert("HSV"))

fig, axes = plt.subplots(1, 4, figsize=(12, 4))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("grayscale (L)")
for i, (name, cmap) in enumerate([("hue", "hsv"), ("saturation", "gray"), ("value", "gray")]):
    axes[i + 1].imshow(hsv[:, :, i], cmap=cmap, vmin=0, vmax=255)
    axes[i + 1].set_title(name)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

The saturation channel shows where the vivid colors are, and the value channel looks much like
the grayscale version. Module 03 builds measures of brightness, saturation and colorfulness on
exactly these channels.

> **✏️ Exercise 2**
>
> 1. Compute the grayscale version of `arr` **by hand** with the formula above, and compare it with
>    Pillow's `gray`. How large is the biggest difference between the two?
>    *(Hint: convert `arr` to floats first with `arr.astype(float)`, and round your result.)*
> 2. Report the image's average brightness (the mean of the grayscale image) and its average
>    saturation (the mean of the HSV saturation channel).

In [ ]:
# ✅ Solution
arr_f = arr.astype(float)
manual_gray = 0.299 * arr_f[:, :, 0] + 0.587 * arr_f[:, :, 1] + 0.114 * arr_f[:, :, 2]
pillow_gray = np.asarray(gray).astype(float)

print("Largest difference:", np.abs(np.round(manual_gray) - pillow_gray).max())
print(f"Average brightness: {pillow_gray.mean():.1f} (0 = black, 255 = white)")
print(f"Average saturation: {hsv[:, :, 1].mean():.1f} (0 = gray, 255 = pure color)")

# The difference is 0 (on some images it can be 1, from rounding): Pillow uses exactly these
# weights. So "grayscale" is not a mysterious transformation, just a weighted average you can
# write down in a methods section.

## 3. Changing the numbers changes the picture

Because an image is an array, editing it is arithmetic. Let's brighten the portrait by adding 100
to every value.

In [ ]:
too_bright = arr + 100          # looks reasonable, but...

fig, axes = plt.subplots(1, 2, figsize=(7, 4.3))
axes[0].imshow(arr)
axes[0].set_title("original")
axes[1].imshow(too_bright)
axes[1].set_title("arr + 100")
for ax in axes:
    ax.axis("off")
plt.show()

print("A pixel before:", arr[300, 225], "and after:", too_bright[300, 225])

That is not a brighter portrait. Because the array is `uint8`, values can't go above 255, and
anything that would wraps around to the bottom: 230 + 100 becomes 74, not 330. NumPy does this
silently. The fix is to compute with a larger integer type, **clip** the result to the 0 to 255
range, and convert back.

In [ ]:
brighter = np.clip(arr.astype(int) + 100, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(7, 4.3))
axes[0].imshow(arr)
axes[0].set_title("original")
axes[1].imshow(brighter)
axes[1].set_title("clipped: brighter, but bright areas burn out")
for ax in axes:
    ax.axis("off")
plt.show()

Cropping and flipping are just indexing. A crop selects a range of rows and columns, and `::-1`
reverses an axis. Pillow has methods for the same operations, which is handy when you don't want
to leave the `Image` object. Note that Pillow's `crop` takes a box of (left, upper, right, lower)
in x, y coordinates, so the order is again the reverse of NumPy's.

In [ ]:
crop_np = arr[0:275, 100:350]                 # rows (y) first, then columns (x)
mirror_np = arr[:, ::-1]                      # reverse the columns
crop_pil = portrait.crop((100, 0, 350, 275))  # (left, upper, right, lower)

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, im, title in zip(axes, [crop_np, mirror_np, crop_pil], ["NumPy crop", "NumPy mirror", "Pillow crop"]):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis("off")
plt.show()

print("Same crop both ways?", np.array_equal(crop_np, np.asarray(crop_pil)))

Resizing is different: it can't be done by indexing alone, because new pixel values have to be
computed from the old ones. Shrinking an image throws information away, and enlarging it again
does not bring that information back.

In [ ]:
small = portrait.resize((90, 110), Image.Resampling.LANCZOS)
back_up = small.resize(portrait.size, Image.Resampling.LANCZOS)
blocky = small.resize(portrait.size, Image.Resampling.NEAREST)

fig, axes = plt.subplots(1, 3, figsize=(10, 4.3))
for ax, im, title in zip(axes, [portrait, back_up, blocky],
                         ["original 450 × 550", "90 × 110, enlarged (smooth)", "90 × 110, enlarged (nearest)"]):
    ax.imshow(im)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.show()

> **✏️ Exercise 3**
>
> 1. Make a **negative** of the portrait (every value becomes 255 minus itself) and a version that
>    keeps only the blue channel (set red and green to 0). Show both next to the original.
> 2. Mirror the portrait with Pillow instead of NumPy (look up `ImageOps.mirror`) and check that the
>    result is identical to `mirror_np`.

In [ ]:
# ✅ Solution
negative = 255 - arr                      # safe: 255 - x never leaves the 0-255 range
blue_only = arr.copy()
blue_only[:, :, 0] = 0
blue_only[:, :, 1] = 0

fig, axes = plt.subplots(1, 3, figsize=(10, 4.3))
for ax, im, title in zip(axes, [arr, negative, blue_only], ["original", "negative", "blue only"]):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis("off")
plt.show()

mirror_pil = ImageOps.mirror(portrait)
print("Pillow and NumPy mirror identical?", np.array_equal(np.asarray(mirror_pil), mirror_np))
# Note the .copy(): without it, blue_only would share memory with arr, and setting channels to 0
# would change the original array too.

## 4. Resolution and file formats

The portrait project offers each photo in several sizes. Let's compare Senator Cantwell's
"original" file with the 450 × 550 version we have been using.

In [ ]:
original = load_image(PHOTO_URL.format(size="original", bioguide="C000127"))
print("Original file:", original.size)
print("450x550 file: ", portrait.size)

The "original" is only 230 × 281 pixels, smaller than the 450 × 550 version, so the larger
version was made by enlarging it: more pixels, but no more detail. Other members' originals are
much larger. Compare her portrait with Senator Amy Klobuchar's (`K000367`), whose original is
675 × 825, both shown at 450 × 550:

In [ ]:
klobuchar = load_image(PHOTO_URL.format(size="450x550", bioguide="K000367"))

fig, axes = plt.subplots(1, 2, figsize=(8, 5))
for ax, im, title in zip(axes, [portrait, klobuchar], ["C000127: enlarged from 230 × 281", "K000367: reduced from 675 × 825"]):
    ax.imshow(np.asarray(im)[120:320, 125:325], interpolation="nearest")   # same region of both
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.show()

This matters for research. A collection that looks uniform (every file 450 × 550) can hide large
differences in true resolution, and anything you measure about sharpness or fine detail will pick
those up. **Check where your images come from and how they were processed before you compare
them.**

The file format matters too. **PNG** stores every pixel exactly (lossless). **JPEG** makes files
much smaller by throwing away detail our eyes are not very sensitive to (lossy), and you choose
how much with a quality setting. Let's save the portrait at a few quality levels and measure the
file size and how far the pixels move from the original.

In [ ]:
def encode(image, fmt, **kwargs):
    """Save an image to memory and return (number of bytes, image read back from those bytes)."""
    buffer = io.BytesIO()
    image.save(buffer, format=fmt, **kwargs)
    n_bytes = buffer.tell()
    buffer.seek(0)
    return n_bytes, Image.open(buffer).convert("RGB")

arr_f = arr.astype(float)
for fmt, kwargs in [("PNG", {}), ("JPEG", {"quality": 95}), ("JPEG", {"quality": 50}), ("JPEG", {"quality": 5})]:
    n_bytes, decoded = encode(portrait, fmt, **kwargs)
    error = np.abs(np.asarray(decoded).astype(float) - arr_f).mean()
    label = fmt if fmt == "PNG" else f"JPEG q={kwargs['quality']}"
    print(f"{label:<12} {n_bytes / 1024:7.1f} KB   average pixel change: {error:5.2f}")

In [ ]:
_, jpeg_5 = encode(portrait, "JPEG", quality=5)

fig, axes = plt.subplots(1, 2, figsize=(8, 4.5))
axes[0].imshow(arr[150:280, 160:290], interpolation="nearest")
axes[0].set_title("original")
axes[1].imshow(np.asarray(jpeg_5)[150:280, 160:290], interpolation="nearest")
axes[1].set_title("JPEG quality 5: blocks and banding")
for ax in axes:
    ax.axis("off")
plt.show()

At low quality, JPEG's 8 × 8 pixel blocks become visible. Most images you collect from the web
have already been compressed, often several times (every re-upload can compress again), so
small differences in measured detail or color between images can come from compression rather
than from what the photo shows.

> **✏️ Exercise 4**
>
> Save the portrait as JPEG at every quality from 5 to 95 in steps of 5. For each, record the file
> size and the average pixel change, then plot both against quality (two small plots side by side).
> Where is the "knee" beyond which a higher quality costs a lot of extra space for little gain?

In [ ]:
# ✅ Solution
qualities = list(range(5, 100, 5))
sizes, errors = [], []
for q in qualities:
    n_bytes, decoded = encode(portrait, "JPEG", quality=q)
    sizes.append(n_bytes / 1024)
    errors.append(np.abs(np.asarray(decoded).astype(float) - arr_f).mean())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(qualities, sizes, marker="o")
axes[0].set_xlabel("JPEG quality")
axes[0].set_ylabel("file size (KB)")
axes[1].plot(qualities, errors, marker="o", color="tab:orange")
axes[1].set_xlabel("JPEG quality")
axes[1].set_ylabel("average pixel change")
plt.tight_layout()
plt.show()

# File size grows slowly up to about quality 80-90 and then shoots up, while the error keeps
# falling only a little. Quality 85-90 is a common sweet spot for storing a research corpus.
# Better still: keep the files exactly as you downloaded them, and never re-save them as JPEG
# just to resize or crop; do that in memory when you analyze them.

## 5. Metadata: what else is in the file

Image files can carry **EXIF** metadata: information written by the camera or the editing
software. Pillow reads it with `getexif()`, which returns numeric tags that `ExifTags.TAGS`
translates into names. Let's look at the original file of Senator Cantwell's portrait.

In [ ]:
exif = original.getexif()
for tag_id, value in exif.items():
    print(f"{ExifTags.TAGS.get(tag_id, tag_id):<26} {value}")

The metadata says the file was saved with Adobe Photoshop CS3 on a Mac in February 2009, and that
the image is 675 × 825 pixels, although the file we downloaded is 230 × 281. Metadata is often
**stale**: it describes an earlier version of the file and was never updated when the image was
resized. Useful as a clue, risky as a fact.

A photo straight from a phone usually carries much more: the phone model, the exact time, and
often **GPS coordinates** of where it was taken. To see what that looks like without using
anyone's real photo, we make a fake phone photo from the portrait. Like many phones, it stores
the pixels sideways and adds an **orientation** tag that says how to rotate them for display.
We add a phone model, a time and the coordinates of Wageningen, and then read everything back as
if we had found the file online.

In [ ]:
phone_exif = Image.Exif()
phone_exif[ExifTags.Base.Make] = "ExamplePhone"
phone_exif[ExifTags.Base.Model] = "Model 12"
phone_exif[ExifTags.Base.DateTime] = "2026:05:14 09:41:07"
phone_exif[ExifTags.Base.Orientation] = 6      # "rotate 90 degrees to display correctly"

gps = phone_exif.get_ifd(ExifTags.IFD.GPSInfo)
gps[ExifTags.GPS.GPSLatitudeRef] = "N"
gps[ExifTags.GPS.GPSLatitude] = (51.0, 58.0, 8.0)     # degrees, minutes, seconds
gps[ExifTags.GPS.GPSLongitudeRef] = "E"
gps[ExifTags.GPS.GPSLongitude] = (5.0, 39.0, 47.0)

sideways = portrait.transpose(Image.Transpose.ROTATE_90)   # store the pixels sideways, as phones do
sideways.save("phone_photo.jpg", quality=95, exif=phone_exif)
print("Saved phone_photo.jpg")

In [ ]:
def gps_to_decimal(dms, ref):
    """Convert (degrees, minutes, seconds) to decimal degrees."""
    degrees, minutes, seconds = (float(x) for x in dms)
    value = degrees + minutes / 60 + seconds / 3600
    return -value if ref in ("S", "W") else value

found = Image.open("phone_photo.jpg")
found_exif = found.getexif()
for tag_id, value in found_exif.items():
    print(f"{ExifTags.TAGS.get(tag_id, tag_id):<12} {value}")

gps_found = found_exif.get_ifd(ExifTags.IFD.GPSInfo)
lat = gps_to_decimal(gps_found[ExifTags.GPS.GPSLatitude], gps_found[ExifTags.GPS.GPSLatitudeRef])
lon = gps_to_decimal(gps_found[ExifTags.GPS.GPSLongitude], gps_found[ExifTags.GPS.GPSLongitudeRef])
print(f"\nTaken at {lat:.4f}, {lon:.4f}: https://www.openstreetmap.org/?mlat={lat:.4f}&mlon={lon:.4f}")

Two lessons. First, an image file can reveal where and when it was taken, and by whom. Many
platforms strip this on upload, but files from archives, messaging apps, email attachments or
data donations often keep it. If you collect images from people, treat metadata as personal data,
and remove it before you share anything.

Second, the **orientation** tag. Photo viewers obey it, so people see the photo upright. Many
analysis pipelines ignore it, and then quietly analyze a sideways image. `ImageOps.exif_transpose`
applies the rotation to the pixels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4.3))
axes[0].imshow(found)
axes[0].set_title(f"pixels as stored {found.size}")
axes[1].imshow(ImageOps.exif_transpose(found))
axes[1].set_title("after exif_transpose")
for ax in axes:
    ax.axis("off")
plt.show()

A face detector run on the stored pixels would be looking for a face lying on its side, and might
not find one. The image your code analyzes is not necessarily the image a person saw, so apply
the orientation tag as one of the first steps whenever you load images from phones or from
people.

> **✏️ Exercise 5**
>
> Write a function `strip_metadata(in_path, out_path)` that:
>
> 1. opens the image and applies the orientation tag, so the pixels end up the way a viewer would
>    display them, and
> 2. saves a copy **without any metadata** (JPEG, quality 95).
>
> Run it on `phone_photo.jpg` and check that the copy has no EXIF data left and is displayed the
> same way as `ImageOps.exif_transpose(found)`.
> *(Hint: `Image.fromarray(np.asarray(img))` creates a new image from the pixels alone.)*

In [ ]:
# ✅ Solution
def strip_metadata(in_path, out_path):
    """Save a copy of an image with the orientation applied and all metadata removed."""
    with Image.open(in_path) as img:
        upright = ImageOps.exif_transpose(img)
        clean = Image.fromarray(np.asarray(upright.convert("RGB")))   # pixels only
        clean.save(out_path, format="JPEG", quality=95)

strip_metadata("phone_photo.jpg", "phone_photo_clean.jpg")
clean = Image.open("phone_photo_clean.jpg")

print("EXIF tags left:", dict(clean.getexif()))
print("Size:", clean.size, "| displayed size of the original:", ImageOps.exif_transpose(found).size)
plt.figure(figsize=(3, 4))
plt.imshow(clean)
plt.axis("off")
plt.show()
# The copy has no metadata, and its pixels are stored the way the original was meant to be shown,
# so every tool will now see the same image. Pillow's save() does not copy EXIF unless you pass
# exif=..., but building the image from bare pixels makes that explicit.

## 6. From one image to a collection

Research rarely involves one image. Let's build a small collection: the biographical data of all
current members of Congress, and the portraits of a random sample of them.

The biographical data is a JSON file with one record per member. We keep a few fields and use the
member's most recent term for chamber, party and state.

In [ ]:
LEGISLATORS_URL = "https://raw.githubusercontent.com/unitedstates/congress-legislators/gh-pages/legislators-current.json"
legislators = download(LEGISLATORS_URL).json()

rows = []
for person in legislators:
    last_term = person["terms"][-1]
    rows.append({
        "bioguide": person["id"]["bioguide"],
        "name": person["name"].get("official_full", f"{person['name']['first']} {person['name']['last']}"),
        "chamber": {"sen": "Senate", "rep": "House"}[last_term["type"]],
        "party": last_term.get("party"),
        "state": last_term["state"],
        "gender": person["bio"].get("gender"),
        "first_year": int(person["terms"][0]["start"][:4]),
    })
members = pd.DataFrame(rows)

print(f"{len(members)} current members")
members.head()

Now we download the small (225 × 275) portraits of a random sample. A few habits are worth
building from the first download on:

- **Save files to disk** and skip the ones you already have, so you never download twice.
- **Pause** briefly between requests, so you don't hammer the server.
- **Record failures** instead of crashing on the first missing file.

200 portraits take a minute or two. Increase `N_PORTRAITS` to `len(members)` if you want them all.

In [ ]:
N_PORTRAITS = 200
PORTRAIT_DIR = Path("portraits")
PORTRAIT_DIR.mkdir(exist_ok=True)

sample = members.sample(N_PORTRAITS, random_state=42).reset_index(drop=True)

missing = []
for i, bioguide in enumerate(sample["bioguide"]):
    path = PORTRAIT_DIR / f"{bioguide}.jpg"
    if path.exists():
        continue
    try:
        path.write_bytes(download(PHOTO_URL.format(size="225x275", bioguide=bioguide)).content)
    except requests.HTTPError:
        missing.append(bioguide)
    time.sleep(0.05)
    if (i + 1) % 50 == 0:
        print(f"{i + 1} of {N_PORTRAITS} done")

sample = sample[~sample["bioguide"].isin(missing)].reset_index(drop=True)
print(f"Downloaded {len(sample)} portraits; {len(missing)} not available")

Next we load all portraits into one NumPy array of shape (number of images, height, width,
channels). This only works if every image has the same size and mode, so we check that as we go.

In [ ]:
images = []
for bioguide in sample["bioguide"]:
    img = Image.open(PORTRAIT_DIR / f"{bioguide}.jpg").convert("RGB")
    if img.size != (225, 275):
        img = img.resize((225, 275), Image.Resampling.LANCZOS)
    images.append(np.asarray(img))
stack = np.stack(images)

print("Stack shape:", stack.shape)
print(f"Memory: {stack.nbytes / 1e6:.1f} MB")

A **contact sheet**, a grid of thumbnails, is the quickest way to get a feel for a collection.
Always look at your images before you measure anything.

In [ ]:
def contact_sheet(images, labels, n_cols=10, size=1.3):
    """Show images in a grid with a small label under each."""
    n_rows = int(np.ceil(len(images) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * size, n_rows * size * 1.35))
    for ax in axes.flat:
        ax.axis("off")
    for ax, img, label in zip(axes.flat, images, labels):
        ax.imshow(img)
        ax.set_title(label, fontsize=7)
    plt.tight_layout()
    plt.show()

labels = [f"{row.party[0]}-{row.state}" for row in sample.itertuples()]
contact_sheet(stack[:40], labels[:40])

Official portraits follow strict conventions: head and shoulders, centered, dark clothing, often
a flag in the background. One way to see a genre's conventions is to **average** all images,
pixel by pixel. What every image shares stays sharp; what varies blurs away.

In [ ]:
average_portrait = stack.mean(axis=0).astype(np.uint8)

plt.figure(figsize=(3, 3.7))
plt.imshow(average_portrait)
plt.title(f"Average of {len(stack)} portraits", fontsize=10)
plt.axis("off")
plt.show()

Finally, our first measurement: the average brightness of each portrait, added as a column to
the table so we can relate it to information about the members.

In [ ]:
gray_stack = stack.astype(float) @ np.array([0.299, 0.587, 0.114])   # grayscale for every image at once
sample["brightness"] = gray_stack.mean(axis=(1, 2))

sample.boxplot(column="brightness", by="chamber", figsize=(5, 3.5))
plt.suptitle("")
plt.title("Average brightness of the portrait")
plt.ylabel("brightness (0-255)")
plt.show()

sample.groupby("chamber")["brightness"].describe().round(1)

> **✏️ Exercise 6**
>
> Make the average portrait separately for the **House** and the **Senate** and show them side by
> side, with the number of portraits in each title. What differs between the two, and what do you
> think explains it?

In [ ]:
# ✅ Solution
fig, axes = plt.subplots(1, 2, figsize=(6, 3.8))
for ax, chamber in zip(axes, ["House", "Senate"]):
    mask = (sample["chamber"] == chamber).to_numpy()
    ax.imshow(stack[mask].mean(axis=0).astype(np.uint8))
    ax.set_title(f"{chamber} (n = {mask.sum()})", fontsize=10)
    ax.axis("off")
plt.show()

# Both averages show the same pose and framing: the conventions of the genre. The backdrops and
# overall tone differ a little, which reflects how and when the photos were taken (studios,
# photographers, the years members took office), and the Senate average rests on far fewer
# images, so it is noisier. None of this says anything about the members themselves; it says how
# the photos were produced. The contact sheet shows one convention clearly: many recent portraits
# have a US flag on the left.

> **✏️ Exercise 7**
>
> Does the brightness of a portrait depend on **when the member first entered Congress**? Make a
> scatter plot of `brightness` against `first_year` and compute the correlation. Then think about
> what this measure really captures: if you found a difference in brightness between Democrats and
> Republicans, how would you know whether it says something about the parties or something about
> the photos?

In [ ]:
# ✅ Solution
ax = sample.plot.scatter(x="first_year", y="brightness", alpha=0.6, figsize=(5, 3.5))
ax.set_xlabel("first year in Congress")
ax.set_ylabel("portrait brightness")
plt.show()

print(f"Correlation: {sample['first_year'].corr(sample['brightness']):.2f}")
print(sample.groupby("party")["brightness"].mean().round(1))

# The relationship is not a straight line: in our sample the darkest portraits cluster among
# members who entered Congress around 2017 to 2021, which suggests a photography style of those
# years (darker backdrops, often with a flag). Long-serving members tend to have older portraits,
# taken with other cameras, lighting and backdrops, and stored at lower resolution (remember
# section 4). Any trend over "first year" is therefore at least partly a trend in photography,
# not in politicians. The same goes for party:
# if one party has more long-serving members, or more Senators, a brightness gap can arise from
# production conditions alone. Before interpreting a visual measure, ask what process produced
# the images, and compare like with like (same year, same chamber, same photographer).

## 7. What can pixels tell a social scientist?

Researchers have used images to study protest and mobilization, how politicians present
themselves, who appears in the news and how, health messages and food marketing, misinformation,
and much more. It helps to separate three levels of what you might want to know about an image:

1. **Content: what is in it.** People, objects, settings, text. "Does this photo show a crowd?
   Is there a flag? How many people are smiling?" Computers have become very good at this level.
2. **Form: how it looks.** Color, brightness, composition, framing, camera angle, resolution.
   Easy to compute (you just did) but often hard to interpret, as the brightness exercise showed.
3. **Meaning: what it communicates.** Visual frames, emotions, symbolism, persuasion. "Does this
   image frame protesters as peaceful or as a threat?" Here, computer output is at best a
   measurement that you must validate against human judgment, and sometimes a careful
   qualitative reading is the better method.

Most good computational image research combines levels: it uses the computer for content and
form at a scale no human team could manage, and links those measures to meaning through a
codebook and validation, just as in text analysis.

> **✏️ Exercise 8**
>
> Think of a collection of images or videos from your own research area (or one you would like to
> study).
>
> 1. What is your **unit of analysis**: a single image, a post with several images and a caption, a
>    video, a person in an image?
> 2. Which of the three levels does your main question sit at: content, form or meaning?
> 3. What could a computer reasonably measure for you, and what would need a human coder or a
>    qualitative reading?
> 4. Where do your images come from, and what processing (cropping, compression, filters, platform
>    re-encoding) might they have gone through before reaching you?

**✅ One possible answer**

*Example: Instagram posts by health influencers about nutrition.*

1. The unit is the **post**: one or more images plus a caption, posted by an account at a time.
   Images within a post are a second level of analysis.
2. The question "how do influencers make nutrition advice look trustworthy?" sits mostly at the
   level of **meaning** (credibility cues, expertise framing), measured through **content** (is
   the influencer in the frame, in a lab coat, in a kitchen; is there text with numbers or
   claims) and **form** (polished studio look versus casual phone photo).
3. A computer can detect people, text, settings and visual style at scale. Whether an image
   *signals* expertise is a judgment that needs a codebook, human coders and validation, and a
   close reading of a smaller sample would help develop that codebook.
4. Instagram crops, compresses and re-encodes uploads, and many posts use filters. That makes
   color and sharpness measures partly a measure of the platform and the filter, not only of
   the content.

## Wrap-up

You have seen that:

- an image is an array of numbers: height × width × channels, each value from 0 to 255;
- color can be represented in different ways (RGB, grayscale, HSV), and the choice decides what is
  easy to measure;
- editing an image is arithmetic, but `uint8` values silently wrap around unless you clip them;
- resolution and JPEG compression vary across a collection, and files can be enlarged copies of
  much smaller originals;
- image files carry metadata that can be stale, revealing or both, and orientation tags can make
  your code see a different image than a person does;
- averaging and contact sheets show a collection's conventions at a glance, and simple measures
  like brightness often reflect how images were produced rather than what they show.

**Next:** in module 02 you build a research corpus from open sources, keep track of licenses, and
deal with duplicates.